In [ ]:
import os
import glob
import math
import time
import re
import json
import random
from collections import Counter

# --- TỰ ĐỘNG QUÉT VÀ CÀI ĐẶT OFFLINE ---
# Tìm chính xác đường dẫn của các file whl trong thư mục input mà không sợ sai tên folder
portalocker_whl = glob.glob('/kaggle/input/**/portalocker-*.whl', recursive=True)
sacrebleu_whl = glob.glob('/kaggle/input/**/sacrebleu-*.whl', recursive=True)

if portalocker_whl and sacrebleu_whl:
    print(f"Tìm thấy Portalocker tại: {portalocker_whl[0]}")
    print(f"Tìm thấy SacreBLEU tại: {sacrebleu_whl[0]}")
    
    # Thực hiện cài đặt lệnh pip offline trực tiếp bằng file tìm được
    os.system(f"pip install {portalocker_whl[0]} -q")
    os.system(f"pip install {sacrebleu_whl[0]} -q")
else:
    print("CẢNH BÁO: Không tìm thấy file .whl trong tập dữ liệu đầu vào!")

# --- TIẾP TỤC IMPORT ---
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import sacrebleu

# Thiết lập seed để đảm bảo kết quả thực nghiệm có thể tái lập
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Khởi tạo cấu trúc thư mục lưu trữ
os.makedirs("processed_data", exist_ok=True)
os.makedirs("checkpoints_base", exist_ok=True)
os.makedirs("checkpoints_big", exist_ok=True)

print("Setup môi trường và tự động nhận diện gói OFFLINE hoàn tất!")

In [ ]:
import os
import glob
import re

def clean_and_tokenize_text(text):
    """
    Tiền xử lý bám sát bài báo PhoMT: Hạ chữ thường, tách toàn bộ dấu câu bằng khoảng trắng.
    """
    text = text.lower().strip()
    # Tách dấu câu bằng khoảng trắng sử dụng Regular Expression
    text = re.sub(re.compile(r'([.,!?"():;])'), r' \1 ', text)
    # Thu gọn nhiều khoảng trắng liên tiếp thành 1 khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_pipeline(output_dir):
    modes = ['train', 'val', 'test']
    langs = ['en', 'vi']
    
    # Quét toàn bộ file trong /kaggle/input để tự động tìm đường dẫn tuyệt đối chính xác của Kaggle
    all_files = glob.glob("/kaggle/input/**/*", recursive=True)
    
    file_map = {m: {l: None for l in langs} for m in modes}
    
    for f_path in all_files:
        if os.path.isfile(f_path):
            filename = os.path.basename(f_path).lower()
            for mode in modes:
                for lang in langs:
                    # Kiểm tra xem tên file có chứa cụm dạng 'train.en' hoặc 'train' và '.en' không
                    if mode in filename and f".{lang}" in filename:
                        file_map[mode][lang] = f_path

    # Hiển thị đường dẫn hệ thống tìm được để kiểm tra
    print("--- Đường dẫn thực tế tìm thấy trên hệ thống Kaggle ---")
    for m in modes:
        for l in langs:
            print(f" -> [{m.upper()} - {l.upper()}]: {file_map[m][l]}")
            if file_map[m][l] is None or not os.path.exists(file_map[m][l]):
                raise FileNotFoundError(f"Hệ thống không thể định vị được file cho tập {m} ngôn ngữ {l}!")

    # Đọc và xử lý loại bỏ trùng lặp chéo trên tập Train
    train_pairs = set()
    with open(file_map['train']['en'], "r", encoding="utf-8") as f_en, \
         open(file_map['train']['vi'], "r", encoding="utf-8") as f_vi:
        for en_line, vi_line in zip(f_en, f_vi):
            en_clean = clean_and_tokenize_text(en_line)
            vi_clean = clean_and_tokenize_text(vi_line)
            # Loại bỏ câu rác, câu trống hoặc câu chỉ có 1 từ theo tiêu chí bài báo
            if en_clean and vi_clean and len(en_clean.split()) > 1:
                train_pairs.add((en_clean, vi_clean))
                
    # Ghi tập Train đã được lọc sạch
    with open(f"{output_dir}/train.clean.en", "w", encoding="utf-8") as f_en, \
         open(f"{output_dir}/train.clean.vi", "w", encoding="utf-8") as f_vi:
        for en_c, vi_c in train_pairs:
            f_en.write(en_c + "\n")
            f_vi.write(vi_c + "\n")
            
    # Xử lý tương tự cho tập Validation và tập Test
    for mode in ['val', 'test']:
        for lang in langs:
            with open(file_map[mode][lang], "r", encoding="utf-8") as f_in, \
                 open(f"{output_dir}/{mode}.clean.{lang}", "w", encoding="utf-8") as f_out:
                for line in f_in:
                    f_out.write(clean_and_tokenize_text(line) + "\n")

    # BỔ SUNG QUAN TRỌNG: Tạo file gộp joint_train.txt phục vụ SentencePiece học Joint BPE
    print("Đang tiến hành tạo file gộp joint_train.txt...")
    with open(f"{output_dir}/joint_train.txt", "w", encoding="utf-8") as f_joint:
        for en_c, vi_c in train_pairs:
            f_joint.write(en_c + "\n")
            f_joint.write(vi_c + "\n")
    print("-> Đã tạo xong file gộp joint_train.txt phục vụ SentencePiece học Joint BPE!")

# Khởi chạy tiền xử lý đầu ra lưu tại folder tạm thời working
preprocess_pipeline("processed_data")
print("Tiền xử lý và tách dấu câu hoàn tất thành công!")

In [ ]:
!pip install --no-index --find-links=/kaggle/input/deep-past-challenge-offline-nlp-wheels/Wheels/ sentencepiece

In [ ]:
import sentencepiece as spm
print(spm.__version__)

In [ ]:
!ls -lh processed_data

In [ ]:
import sentencepiece as spm
import os

# SỬA DÒNG NÀY: Thêm flush=True để ép Kaggle in ra màn hình ngay lập tức!
print("--- KHỞI CHẠY BPE SIÊU TỐC VỚI SENTENCEPIECE & LƯU OUTPUT ---", flush=True)

# Đảm bảo thư mục đầu ra tồn tại trong vùng không gian Output vật lý
os.makedirs("processed_data", exist_ok=True)

# 1. Huấn luyện Joint BPE model chung cho cả 2 ngôn ngữ và lưu thẳng model ra output
spm.SentencePieceTrainer.train(
    input="processed_data/joint_train.txt",
    model_prefix="processed_data/spm_bpe",
    vocab_size=10000,
    model_type="bpe",
    character_coverage=1.0,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    pad_piece="<pad>", unk_piece="<unk>", bos_piece="<sos>", eos_piece="<eos>"
)

# Nạp model từ thư mục output để tiến hành mã hóa dữ liệu
sp = spm.SentencePieceProcessor(model_file="processed_data/spm_bpe.model")
print("🎉 Huấn luyện và lưu trữ thành công SentencePiece Model ra Output!")

def encode_file(input_path, output_path):
    with open(input_path, "r", encoding="utf-8") as f_in, \
         open(output_path, "w", encoding="utf-8") as f_out:
        for line in f_in:
            # Mã hóa chuỗi text thành các mảnh subwords cách nhau bằng dấu cách
            pieces = sp.encode_as_pieces(line.strip())
            f_out.write(" ".join(pieces) + "\n")

# 2. Áp dụng (Apply) và ghi trực tiếp các file .bpe vật lý ra Output
print("Đang tiến hành áp dụng BPE và xuất file ra thư mục Output...")
pair_data = [
    ("train.clean.en", "train.bpe.en"), ("train.clean.vi", "train.bpe.vi"),
    ("val.clean.en", "val.bpe.en"), ("val.clean.vi", "val.bpe.vi"),
    ("test.clean.en", "test.bpe.en"), ("test.clean.vi", "test.bpe.vi")
]

for src, trg in pair_data:
    out_path = f"processed_data/{trg}"
    encode_file(f"processed_data/{src}", out_path)
    print(f" -> Đã lưu file BPE vật lý: {out_path}")
    
print("✨ HOÀN THÀNH CELL 3: Toàn bộ dữ liệu BPE đã nằm an toàn trong Output!")

In [ ]:
import json
import sentencepiece as spm
from collections import Counter
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

print("--- [CELL 4] SỐ HÓA TỪ VỰNG & CẮT ĐUÔI CÂU DÀI CHỐNG OOM ---", flush=True)

# Nạp bộ não SentencePiece mẫu
sp = spm.SentencePieceProcessor(model_file="processed_data/spm_bpe.model")

class Vocabulary:
    def __init__(self, sp_processor):
        self.sp = sp_processor
        self.str2id = {}
        self.id2str = {}
        self.build_vocab()
        
    def build_vocab(self):
        for i in range(self.sp.get_piece_size()):
            piece = self.sp.id_to_piece(i)
            self.str2id[piece] = i
            self.id2str[i] = piece

    def numericalize(self, text):
        return self.sp.encode_as_ids(text.strip())

    def save_vocab(self, json_path):
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump({"str2id": self.str2id, "id2str": {str(k): v for k, v in self.id2str.items()}}, f, ensure_ascii=False, indent=4)
        print(f" -> Đã lưu bộ từ vựng vật lý vào: {json_path}", flush=True)

    def __len__(self):
        return self.sp.get_piece_size()

# Khởi tạo duy nhất bộ từ vựng Joint
vocab_joint = Vocabulary(sp)
vocab_joint.save_vocab("processed_data/vocab_joint.json")

class PhomtDataset(Dataset):
    def __init__(self, src_path, trg_path, vocab_joint, max_len=128): # GIỚI HẠN ĐỘ DÀI CÂU TỐI ĐA
        self.src_data = []
        self.trg_data = []
        self.vocab = vocab_joint # <── ĐÃ SỬA: Đồng bộ đúng tên tham số truyền vào!
        
        with open(src_path, "r", encoding="utf-8") as f_src, \
             open(trg_path, "r", encoding="utf-8") as f_trg:
            for src_line, trg_line in zip(f_src, f_trg):
                src_ids = self.vocab.numericalize(src_line)
                trg_ids = self.vocab.numericalize(trg_line)
                
                if len(src_ids) == 0 or len(trg_ids) == 0:
                    continue
                    
                # Cắt đuôi chuỗi nếu câu vượt quá 128 từ để bảo vệ VRAM
                src_ids = src_ids[:max_len]
                trg_ids = trg_ids[:max_len]
                
                self.src_data.append(torch.tensor(src_ids, dtype=torch.long))
                self.trg_data.append(torch.tensor(trg_ids, dtype=torch.long))
                
    def __len__(self):
        return len(self.src_data)
    
    def __getitem__(self, idx):
        return self.src_data[idx], self.trg_data[idx]

def pad_collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    src_padded = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    trg_padded = nn.utils.rnn.pad_sequence(trg_batch, batch_first=True, padding_value=0)
    return src_padded, trg_padded

# ĐÃ SỬA: Xóa bỏ 2 dòng rác trùng lặp cũ mang tên vocab_en. Chỉ giữ lại khai báo vocab_joint chuẩn.
train_dataset = PhomtDataset("processed_data/train.clean.en", "processed_data/train.clean.vi", vocab_joint, max_len=128)
val_dataset = PhomtDataset("processed_data/val.clean.en", "processed_data/val.clean.vi", vocab_joint, max_len=128)

# Đẩy mạnh tốc độ nạp data của đa nhân CPU hỗ trợ RTX 6000
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=pad_collate_fn, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, collate_fn=pad_collate_fn, num_workers=2)

print(f"🎉 Hoàn thành xuất sắc CELL 4! Bộ lọc chống tràn RAM hoạt động tốt và sạch hoàn toàn lỗi NameError.", flush=True)

In [ ]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class PyTorchTransformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, d_model=512, nhead=8, num_layers=6, d_ff=2048, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.pos_decoder = PositionalEncoding(d_model)
        
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, 
            num_encoder_layers=num_layers, num_decoder_layers=num_layers, 
            dim_feedforward=d_ff, dropout=dropout, batch_first=True
        )
        self.fc_out = nn.Linear(d_model, trg_vocab_size)

    def generate_square_subsequent_mask(self, sz, device):
        mask = (torch.triu(torch.ones(sz, sz, device=device)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, src, trg):
        src_mask = None
        trg_mask = self.generate_square_subsequent_mask(trg.size(1), trg.device)
        
        src_padding_mask = (src == 0)
        trg_padding_mask = (trg == 0)
        
        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        trg_emb = self.pos_decoder(self.trg_embedding(trg) * math.sqrt(self.d_model))
        
        out = self.transformer(src_emb, trg_emb, src_mask, trg_mask, None, 
                               src_padding_mask, trg_padding_mask, src_padding_mask)
        return self.fc_out(out)

print("--- [CELL 5] Định nghĩa cấu trúc mạng Transformer PhoMT hoàn tất ---", flush=True)

In [ ]:
import os
import math
import torch
import gc
import pandas as pd  
from tqdm import tqdm
import sacrebleu  

def train_and_validate(model, train_loader, val_loader, optimizer, criterion, epochs, save_dir, sp_processor):
    os.makedirs(save_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    best_val_loss = float('inf')
    steps_per_epoch = len(train_loader)
    
    history = {
        "epoch": [], "train_loss": [], "val_loss": [], "bleu": [], "ter": []
    }
    
    print(f"🚀 HỆ THỐNG ĐÃ CẤU HÌNH LẠI: Tối ưu GPU tuyệt đối, BLEU tính cuối Epoch!", flush=True)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        progress_bar = tqdm(enumerate(train_loader), total=steps_per_epoch, desc=f"Epoch {epoch+1:02d}/{epochs}")
        
        # GIẢI PHÓNG TỐC ĐỘ: Vòng train chỉ thuần túy tính toán ma trận trên GPU
        for step, (src, trg) in progress_bar:
            src, trg = src.to(device), trg.to(device)
            trg_input = trg[:, :-1]
            trg_expected = trg[:, 1:]
            
            optimizer.zero_grad()
            output = model(src, trg_input)
            loss = criterion(output.reshape(-1, output.shape[-1]), trg_expected.reshape(-1))
            loss.backward()
            optimizer.step()
            
            current_loss = loss.item()
            total_loss += current_loss
            
            # Cập nhật thanh tiến độ nhanh, tránh in quá nhiều làm chậm máy
            if step % 10 == 0 or step == steps_per_epoch - 1:
                progress_bar.set_postfix({"Loss": f"{current_loss:.4f}"})
                
            del output, loss
            
        avg_train_loss = total_loss / steps_per_epoch
        
        # --- KHÂU ĐÁNH GIÁ (CHỈ CHẠY 1 LẦN DUY NHẤT KHI HẾT EPOCH) ---
        model.eval()
        val_loss = 0
        hypotheses = []
        references = []
        
        print(f"\n⏳ Đang quét tập Validation và tính điểm BLEU/TER...", end="", flush=True)
        with torch.no_grad():
            for v_idx, (v_src, v_trg) in enumerate(val_loader):
                v_src, v_trg = v_src.to(device), v_trg.to(device)
                v_trg_input = v_trg[:, :-1]
                v_trg_expected = v_trg[:, 1:]
                
                v_output = model(v_src, v_trg_input)
                v_loss = criterion(v_output.reshape(-1, v_output.shape[-1]), v_trg_expected.reshape(-1))
                val_loss += v_loss.item()
                
                # Chỉ giải mã chữ (decode) ở tập Validation với số lượng giới hạn để không gây bottleneck
                if v_idx < 4: # Lấy khoảng 4 batches ~ 512 câu để chấm điểm đại diện
                    preds = v_output.argmax(dim=-1)
                    for p, t in zip(preds, v_trg_expected):
                        hypotheses.append(sp_processor.decode(p.tolist()))
                        references.append([sp_processor.decode(t.tolist())])
                        
                del v_output, v_loss
        
        avg_val_loss = val_loss / len(val_loader)
        
        # Tính chỉ số BLEU và TER sau khi thu thập đủ chữ
        bleu_score = sacrebleu.corpus_bleu(hypotheses, references).score
        ter_score = sacrebleu.corpus_ter(hypotheses, references).score
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), f"{save_dir}/best_model.pt")
            print(f"\n🌟 Cập nhật kỷ lục trọng số mới!")
            
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["bleu"].append(bleu_score)
        history["ter"].append(ter_score)
        
        df_log = pd.DataFrame(history)
        df_log.to_csv(f"{save_dir}/training_log.csv", index=False)
        
        print(f"⇒ TỔNG KẾT EPOCH {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | BLEU: {bleu_score:.2f} | TER: {ter_score:.2f}")
        
        gc.collect()
        torch.cuda.empty_cache()
        
    return history

print("--- [CELL 6] ĐÃ KHÓA BOTTLENECK CHẤM ĐIỂM, GIẢI PHÓNG ĐƯỜNG ĐUA! ---")

In [ ]:
print("\n--- [CELL 7] KHỞI CHẠY TIẾN TRÌNH HUẤN LUYỆN TRANSFORMER-BASE ---", flush=True)

# ĐÃ SỬA: Đồng bộ gọi tên vocab_joint cho cả lớp Embedding nguồn và đích
model_base = PyTorchTransformer(
    src_vocab_size=len(vocab_joint), 
    trg_vocab_size=len(vocab_joint),
    d_model=512, nhead=8, num_layers=6, d_ff=2048, dropout=0.1
)

# ĐÃ SỬA: Ép lr xuống 5e-5 để xoa dịu ma trận, giúp Transformer tự hội tụ không cần Warmup
optimizer_base = torch.optim.Adam(model_base.parameters(), lr=5e-5, betas=(0.9, 0.98), eps=1e-9)

criterion = nn.CrossEntropyLoss(ignore_index=0)

# Kích hoạt vòng lặp huấn luyện mới chuẩn chỉnh ngữ nghĩa
history_base = train_and_validate(
    model_base, train_loader, val_loader, 
    optimizer_base, criterion, epochs=20, save_dir="checkpoints_base",
    sp_processor=sp
)

In [ ]:
# print("\n--- [CELL 8] KHỞI CHẠY TIẾN TRÌNH HUẤN LUYỆN TRANSFORMER-BIG ---", flush=True)
# model_big = PyTorchTransformer(
#     src_vocab_size=len(vocab_en), trg_vocab_size=len(vocab_vi),
#     d_model=1024, nhead=16, num_layers=6, d_ff=4096, dropout=0.1
# )

# optimizer_big = torch.optim.Adam(model_big.parameters(), lr=3e-4, betas=(0.9, 0.98), eps=1e-9)

# # CẬP NHẬT CHUẨN XÁC: Truyền 'sp' vào đúng vị trí cuối cùng của hàm train_and_validate
# history_big = train_and_validate(
#     model_big, 
#     train_loader, 
#     val_loader,
#     optimizer_big, 
#     criterion, 
#     epochs=20,               # Đã hạ xuống 20 Epochs bằng bản base
#     save_dir="checkpoints_big",
#     sp_processor=sp          # Đưa 'sp' vào đúng vị trí cuối cùng của đối số
# )

In [ ]:
# import matplotlib.pyplot as plt
# import sacrebleu
# import torch

# print("\n--- [CELL 9] ĐANG VẼ BIỂU ĐỒ TIẾN TRÌNH HỌC TẬP (LOSS CHART) ---", flush=True)
# plt.figure(figsize=(12, 5))

# plt.subplot(1, 2, 1)
# plt.plot(range(1, len(history_base[0])+1), history_base[0], label='Train Loss', color='blue')
# plt.plot(range(1, len(history_base[1])+1), history_base[1], label='Valid Loss', color='orange')
# plt.title('Transformer-Base Loss Curve')
# plt.xlabel('Epochs')
# plt.ylabel('Loss')
# plt.legend()

# plt.subplot(1, 2, 2)
# plt.plot(range(1, len(history_big[0])+1), history_big[1-1], label='Train Loss', color='blue') # Sửa lỗi chỉ số mảng lịch sử
# plt.plot(range(1, len(history_big[1])+1), history_big[1], label='Valid Loss', color='orange')
# plt.title('Transformer-Big Loss Curve')
# plt.xlabel('Epochs')
# plt.ylabel('Loss')
# plt.legend()

# plt.tight_layout()
# plt.savefig('processed_data/loss_training_chart.png', dpi=300) 
# plt.show()

# print("\n--- ĐANG TIẾN HÀNH GIẢI MÃ BEAM SEARCH TRÊN TẬP TEST ---", flush=True)
# # ĐỒNG BỘ THIẾT BỊ GPU KHI LOAD WEIGHT
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model_base.load_state_dict(torch.load("checkpoints_base/best_model.pt", map_location=device))
# model_big.load_state_dict(torch.load("checkpoints_big/best_model.pt", map_location=device))

# # Đẩy cả 2 model lên GPU sẵn sàng phục vụ Test
# model_base.to(device)
# model_big.to(device)

# test_src_sentences = []
# with open("processed_data/test.bpe.en", "r", encoding="utf-8") as f:
#     for line in f:
#         test_src_sentences.append(vocab_en.numericalize(line))

# references = []
# with open("processed_data/test.clean.vi", "r", encoding="utf-8") as f:
#     for line in f:
#         references.append(line.strip())

# preds_base = []
# preds_big = []

# print("Đang dịch tự động tập Test bằng thuật toán Beam Search...", flush=True)
# for idx, src_ids in enumerate(test_src_sentences):
#     pred_b = beam_search_decode(model_base, src_ids, vocab_vi, beam_size=4, length_penalty=0.6)
#     pred_B = beam_search_decode(model_big, src_ids, vocab_vi, beam_size=4, length_penalty=0.6)
#     preds_base.append(pred_b)
#     preds_big.append(pred_B)

# bleu_base = sacrebleu.corpus_bleu(preds_base, [references], force=True).score
# ter_base = sacrebleu.corpus_ter(preds_base, [references]).score

# bleu_big = sacrebleu.corpus_bleu(preds_big, [references], force=True).score
# ter_big = sacrebleu.corpus_ter(preds_big, [references]).score

# print("\n" + "="*60)
# print(f"{'MÔ HÌNH THỰC NGHIỆM (PhoMT 250K)':<32} | {'BLEU (↑)':<10} | {'TER (↓)':<10}")
# print("="*60)
# print(f"{'Transformer-Base (Cấu hình chuẩn)':<32} | {bleu_base:<10.2f} | {ter_base:<10.2f}")
# print(f"{'Transformer-Big (Mạng nâng cao)':<32} | {bleu_big:<10.2f} | {ter_big:<10.2f}")
# print("="*60, flush=True)

In [ ]:
------------------------------------------------HẾT-----------------------------------------